DSPy 是一个由斯坦福大学推出的框架，旨在将提示工程（Prompt Engineering）转变为编程。它的核心思想是：你定义签名（Signature）和逻辑结构（Module），而让 DSPy 的编译器（Compiler）去自动为你优化提示词。

安装 dspy:

In [1]:
%pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 52.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 60.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 26.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 27.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 50.5 MB/s  0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this wa

In [2]:
import os
api_key = os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

在使用 DSPy 之前，我们需要导入依赖:

In [4]:
import dspy

In [ ]:
lm = dspy.LM(
    "openai/gemini-3-flash-preview",
    api_base=base_url,
    api_key=api_key,
    extra_body={
        "reasoning_effort": "none",
        "chat_template_kwargs": {
            "enable_thinking": False,
        },
        "temperature": 0,
    },
)

dspy.configure(lm=lm)

`dspy.configure(lm=lm)` 是在做什么？

简单来说这是 DSPy 的注册机制，作用是设置全局默认语言模型，如果没有显式指定模型，它们会自动寻找全局配置中的 `lm`。

在 DSPy 中，`dspy.configure` 和 `dspy.context` 都用于管理模型、回调函数等配置，区别是他们的使用场景不同。

`dspy.configure` 是当你希望整个脚本或应用默认使用某一个模型时使用。通常出现在代码的初始化阶段。

`dspy.context` 是局部作用域，仅在 `with` 语句块内有效，例如在多模型流水线，涉及线程安全的场景下使用，例如集成在 LangFlow 的组件。

## Example1: Basic Text Classification
> One input, one output field.

定义签名:

In [6]:
class Sentiment(dspy.Signature):
    """Classify the sentiment of a given sentence."""

    sentence: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")

In [14]:
print(f"要输入的参数 {list(Sentiment.input_fields.keys())}, 定义的模型输出参数 {list(Sentiment.output_fields.keys())}")

要输入的参数 ['sentence'], 定义的模型输出参数 ['sentiment']
